<a href="https://colab.research.google.com/github/deeppatel710/MiniWarehouseWorkshop/blob/main/MiniWarehouseWorkshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MiniWarehouseWorkshop") \
    .getOrCreate()

# Go to terminal and type:
`gh auth login`

In [ ]:
!rm -rf MiniWarehouseWorkshop  # clean if re-running

In [ ]:
!git clone https://github.com/deeppatel710/MiniWarehouseWorkshop.git

In [ ]:
!ls MiniWarehouseWorkshop
!ls MiniWarehouseWorkshop/data

In [ ]:
base_path = "MiniWarehouseWorkshop/data/"
users = spark.read.csv(base_path + "users.csv", header=True, inferSchema=True)
user_fields = spark.read.csv(base_path + "user_fields.csv", header=True, inferSchema=True)
products = spark.read.csv(base_path + "products.csv", header=True, inferSchema=True)
product_categories = spark.read.csv(base_path + "product_categories.csv", header=True, inferSchema=True)
orders = spark.read.csv(base_path + "orders.csv", header=True, inferSchema=True)
order_items = spark.read.csv(base_path + "order_items.csv", header=True, inferSchema=True)

In [ ]:
users.printSchema()
users.show(10)

In [ ]:
products.printSchema()
products.show(10)

In [ ]:

product_categories.printSchema()
product_categories.show(10)

In [ ]:
orders.printSchema()
orders.show(10)

In [ ]:
order_items.printSchema()
order_items.show(10)

### GOAL: Calculate Revenue per Field per Month

OLTP way of doing this:

In [ ]:
from pyspark.sql.functions import col, month, year, sum

oltp_query = (
    order_items
    .join(products, "product_id")
    .join(orders, "order_id")
    .join(users, "user_id")
    .join(user_fields, "field_id")
    .withColumn("year", year("order_date"))
    .withColumn("month", month("order_date"))
    .withColumn("revenue", col("quantity") * col("unit_price"))
    .groupBy("field_name", "year", "month")
    .agg(sum("revenue").alias("total_revenue"))
    .orderBy("field_name", "year", "month")
)

oltp_query.show()


# OLAP WAY : THE ONLY RIGHT WAY

# Cleaning first

In [ ]:
users = users.dropna()

users = users.withColumn(
    "created_at",
    col("created_at").cast("date")
)

users_clean_df = users.dropDuplicates(["user_id"])

In [ ]:
users_clean_df.show(10)

In [ ]:
from pyspark.sql.functions import col

orders = orders.dropna()

orders = orders.withColumn(
    "order_date",
    col("order_date").cast("date")
)

orders_df = orders.dropDuplicates(["order_id"])

In [ ]:
orders_df.show(10)

# We will create:

# ⭐ dim_user
# ⭐ dim_product
# ⭐ dim_date
# ⭐ fact_sales

In [ ]:
from pyspark.sql.functions import concat_ws, monotonically_increasing_id, split

dim_user = (
    users
    .join(user_fields, "field_id")
    .select("user_id", "user_name", "field_name", "country")
    .withColumn("user_sk", monotonically_increasing_id())
)
dim_user.show(10)

In [ ]:
dim_product = (
    products
    .join(product_categories, "category_id")
    .select("product_id", "product_name", "category_name", "unit_price")
    .withColumn("product_sk", monotonically_increasing_id())
)
dim_product.show(10)

In [ ]:
dim_date = (
    orders
    .select("order_date")
    .dropDuplicates()
    .withColumn("year", year("order_date"))
    .withColumn("month", month("order_date"))
    .withColumn("date_sk", monotonically_increasing_id())
)
dim_date.show(10)

In [ ]:
fact_sales = (
    order_items
    .join(orders, "order_id")
    .join(dim_user, "user_id")
    .join(dim_product, "product_id")
    .join(dim_date, "order_date")
    .select(
        "order_id",
        "user_sk",
        "product_sk",
        "date_sk",
        "quantity",
        (col("quantity") * col("unit_price")).alias("revenue")
    )
)
fact_sales.show(10)

# Analytics in Warehouse Layer

In [ ]:
analytics_query = (
    fact_sales
    .join(dim_user, "user_sk")
    .join(dim_date, "date_sk")
    .groupBy("field_name", "year", "month")
    .sum("revenue")
)

analytics_query.show()


# Denormalize Further (BI Layer)

In [ ]:
analytics_flat = (
    fact_sales
    .join(dim_user, "user_sk")
    .join(dim_product, "product_sk")
    .join(dim_date, "date_sk")
)

analytics_flat.show()


# Now analytics becomes:

In [ ]:
analytics_flat.groupBy("category_name", "year").sum("revenue").show()


# by country:

In [ ]:
analytics_flat.groupBy("country", "year").sum("revenue").show()


# DO IT YOURSELF:
Question 1: Which research field generated the most revenue overall?

Question 2: Who are the top 3 scientists by total revenue?